In [ ]:
# Lecture 15 — Reading Data: CSV, JSON, Parquet
# Spark 101 · Module 5 · Lecture 15
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- Stage the same data in three formats on the driver's local disk -------
# We write CSV and JSON by hand, then let Spark write the Parquet copy.
import os

base = "/tmp/spark101_l15"
os.makedirs(base, exist_ok=True)

csv_text = """id,name,city,price
1,Ada,Madrid,19.99
2,Linus,Helsinki,5.50
3,Grace,New York,1200.00
"""
with open(f"{base}/sales.csv", "w") as f:
    f.write(csv_text)

# One JSON object per line — the default Spark expects.
json_text = (
    '{"id": 1, "name": "Ada",   "city": "Madrid",   "price": 19.99}\n'
    '{"id": 2, "name": "Linus", "city": "Helsinki", "price": 5.50}\n'
    '{"id": 3, "name": "Grace", "city": "New York", "price": 1200.00}\n'
)
with open(f"{base}/sales.json", "w") as f:
    f.write(json_text)

print("staged CSV and JSON in", base)

In [ ]:
# --- Read #1: CSV. Needs header + inference; everything else is a string. ---
df_csv = spark.read.csv(f"file:{base}/sales.csv", header=True, inferSchema=True)
df_csv.show()
df_csv.printSchema()
# Long form is identical:
#   spark.read.format("csv").option("header", True).option("inferSchema", True).load(...)

In [ ]:
# --- Read #2: JSON. Self-describing — the field names come for free. --------
df_json = spark.read.json(f"file:{base}/sales.json")
df_json.show()
df_json.printSchema()
# Note: Spark still scans the file to infer types, and reads every field.

In [ ]:
# --- Write the data as Parquet, then read it back. -------------------------
# Parquet bakes the schema into the file — no header, no inference needed.
df_csv.write.mode("overwrite").parquet(f"file:{base}/sales_parquet")
df_parquet = spark.read.parquet(f"file:{base}/sales_parquet")
df_parquet.show()
df_parquet.printSchema()
# printSchema is exact — types were stored, not guessed.

In [ ]:
# --- Columnar payoff: select one column. Parquet reads only that column. ----
# Logically the same result; physically Parquet touched a fraction of the bytes.
df_parquet.select("price").show()

In [ ]:
# --- Size on disk: Parquet vs CSV -----------------------------------------
def dir_size(path):
    total = 0
    for root, _, files in os.walk(path):
        for fn in files:
            total += os.path.getsize(os.path.join(root, fn))
    return total

csv_bytes = os.path.getsize(f"{base}/sales.csv")
pq_bytes = dir_size(f"{base}/sales_parquet")
print(f"CSV bytes:     {csv_bytes}")
print(f"Parquet bytes: {pq_bytes}")
# On tiny data Parquet's footer overhead can make it larger — the win shows up
# at real scale, where columnar compression dominates. Say that on camera.

In [ ]:
# --- mode: how to handle a bad row ----------------------------------------
bad_csv = "id,name,price\n1,Ada,19.99\n2,Linus,not_a_number\n"
with open(f"{base}/bad.csv", "w") as f:
    f.write(bad_csv)

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("price", DoubleType()),
])

# PERMISSIVE (default) — keep going, null the broken field
spark.read.csv(f"file:{base}/bad.csv", header=True, schema=schema, mode="PERMISSIVE").show()
# DROPMALFORMED — throw the bad row away
spark.read.csv(f"file:{base}/bad.csv", header=True, schema=schema, mode="DROPMALFORMED").show()

In [ ]:
# CSV with an EXPLICIT schema — no inference pass, exact types.
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
sales_schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("city", StringType()),
    StructField("price", DoubleType()),
])
spark.read.csv(f"file:{base}/sales.csv", header=True, schema=sales_schema).printSchema()

In [ ]:
# Nested JSON: objects map to structs, arrays to array columns.
nested = '[{"id":1,"customer":{"name":"Ada","vip":true},"items":["a","b"]}]'
with open(f"{base}/nested.json", "w") as f:
    f.write(nested)
dn = spark.read.option("multiLine", True).json(f"file:{base}/nested.json")
dn.printSchema()
dn.show(truncate=False)

In [ ]:
# Columnar read: Parquet touches only the column you select.
df_parquet.select("price").show()

In [ ]:
# A different delimiter — semicolon-separated values.
with open(f"{base}/semi.csv", "w") as f:
    f.write("id;name\n1;Ada\n2;Linus\n")
spark.read.csv(f"file:{base}/semi.csv", header=True, sep=";").show()

In [ ]:
# Recap:
#   - One reader: spark.read.<format>(path) with options.
#   - CSV: readable but typeless and row-based. JSON: self-describing, nested.
#   - Parquet: columnar + typed + compressed — the production default.
# Next lecture: reshaping data — select, filter, groupBy, join.